# 01 — Data Profiling

**Objetivo:** Carregar todas as tabelas brutas e documentar a estrutura, qualidade e cobertura dos dados antes de qualquer transformação.

**Tabelas analisadas:**
- `orders` — 9.999 pedidos de entrega
- `customers` — cadastro de clientes
- `products` — catálogo de produtos
- `drivers` — motoristas de entrega
- `order_items` — itens por pedido (tabela de ligação)

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.data_loader import load_all

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
tables = load_all()

for name, df in tables.items():
    print(f"{'='*50}")
    print(f"Tabela: {name.upper()} | Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]}")
    print(f"{'='*50}")
    print(df.dtypes)
    print()

Tabela: ORDERS | Linhas: 10,000 | Colunas: 9
date               object
order_id           object
order_amount       object
region             object
items_delivered     int64
items_missing       int64
delivery_hour      object
driver_id          object
customer_id        object
dtype: object

Tabela: CUSTOMERS | Linhas: 1,239 | Colunas: 3
customer_id      object
customer_name    object
customer_age      int64
dtype: object

Tabela: PRODUCTS | Linhas: 314 | Colunas: 4
produc_id       object
product_name    object
category        object
price           object
dtype: object

Tabela: DRIVERS | Linhas: 1,247 | Colunas: 4
driver_id      object
driver_name    object
age             int64
Trips           int64
dtype: object

Tabela: ORDER_ITEMS | Linhas: 1,662 | Colunas: 2
order_id      object
product_id    object
dtype: object



## Valores nulos por tabela

In [3]:
for name, df in tables.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if nulls.empty:
        print(f"{name}: sem valores nulos")
    else:
        print(f"\n{name}:")
        print(nulls)

orders: sem valores nulos
customers: sem valores nulos
products: sem valores nulos
drivers: sem valores nulos
order_items: sem valores nulos


## Duplicatas

In [4]:
for name, df in tables.items():
    dups = df.duplicated().sum()
    print(f"{name}: {dups} linha(s) duplicada(s)")

orders: 0 linha(s) duplicada(s)
customers: 0 linha(s) duplicada(s)
products: 0 linha(s) duplicada(s)
drivers: 0 linha(s) duplicada(s)
order_items: 36 linha(s) duplicada(s)


## Amostra de cada tabela

In [5]:
for name, df in tables.items():
    print(f"\n--- {name.upper()} ---")
    display(df.head(3))


--- ORDERS ---


,date,order_id,order_amount,region,items_delivered,items_missing,delivery_hour,driver_id,customer_id
0,2023-01-01,c9da15aa-be24-4871-92a3-dfa7746fff69,"$1,095.54",Winter Park,10,1,8:37:28,WDID10627,WCID5031
1,2023-01-01,ccacc183-09f8-4fd5-af35-009d18656326,$659.11,Altamonte Springs,11,1,9:31:17,WDID10533,WCID5794
2,2023-01-01,f4e1d30b-c3d1-413f-99b8-93c0b46d68bf,$251.45,Winter Park,18,1,10:43:49,WDID10559,WCID5599



--- CUSTOMERS ---


,customer_id,customer_name,customer_age
0,WCID5170,Elijah Taylor,30
1,WCID5901,Alexis Ross,58
2,WCID5652,Carla Knox,23



--- PRODUCTS ---


,produc_id,product_name,category,price
0,PWPX0982761090982,Kellogg's Frosties,Supermarket,$12.53
1,PWPX0982761090983,Uncured Bacon,Supermarket,$4.67
2,PWPX0982761090984,Whole Milk,Supermarket,$9.95



--- DRIVERS ---


,driver_id,driver_name,age,Trips
0,WDID09873,Pamela Moore,18,64
1,WDID09874,Billy Lawson,18,37
2,WDID09875,Stephen Randolph,18,64



--- ORDER_ITEMS ---


,order_id,product_id
0,c7a343f7-3f1d-497c-8004-b9ede2d48fb1,PWPX0982761090982
1,c7a343f7-3f1d-497c-8004-b9ede2d48fb1,PWPX0982761090982
2,20698293-8399-4fda-af1e-b61a9ebb8a0a,PWPX0982761090983


## Problemas identificados

| Tabela | Coluna | Problema | Ação planejada |
|---|---|---|---|
| orders | `order_amount` | Formato string com `$` e `,` | Converter para float |
| orders | `delivery_hour` | String de tempo (`HH:MM:SS`) | Extrair hora como inteiro |
| orders | `date` | String | Converter para datetime |
| products | `produc_id` | Typo no nome da coluna | Renomear para `product_id` |
| products | `price` | Formato string com `$` | Converter para float |
| drivers | `Trips` | Nome em maiúscula | Renomear para `trips` |